### 06. Feature Selection and Recommendation Quality Evaluation

**Objective:** Compare four feature combinations (Title-only, Title+Tags, Title+Description+Tags, Title+Description+Transcript+Tags) and identify which produces the most relevant TED Talk recommendations.

**Input:** `../data/processed/ted_talks_cleaned.csv`
**Approach:** Build a TF-IDF + cosine similarity pipeline per experiment (computed once per experiment, cached in memory), then compare top-5 recommendations for 5 seed talks across all 4 experiments.

**Seeds used:** 
- Averting the climate crisis
- Do schools kill creativity
- Simplicity sells
- The best stats you've ever seen
- How policing for profit undermines your rights

In [14]:
import numpy as np
import pandas as pd
import time
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [15]:
# Load dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print("Available text columns:", [c for c in ["title", "tags", "description", "transcript"] if c in df.columns])
df[["talk_id", "title", "tags", "description", "transcript"]].head(2)

Dataset: 4005 rows, 16 columns
Available text columns: ['title', 'tags', 'description', 'transcript']


,talk_id,title,tags,description,transcript
0,1,averting the climate crisis,"alternative energy, cars, climate change, culture, environment, global issues, science, sustainability, technology",with the same humor and humanity he exuded in an inconvenient truth al gore spells out 15 ways that individuals can ...,thank you so much chris and it s truly a great honor to have the opportunity to come to this stage twice i m extreme...
1,92,the best stats you ve ever seen,"africa, asia, google, demo, economics, global issues, health, statistics, global development, visualizations, math",you ve never seen data presented like this with the drama and urgency of a sportscaster statistics guru hans rosling...,about 10 years ago i took on the task to teach global development to swedish undergraduate students that was after h...


##### Define the Four Experiments
Each experiment builds a TF-IDF matrix and a cosine similarity matrix from a different combination of columns.

In [16]:
def build_tfidf_cosim(text_series):
    """Build TF-IDF matrix and cosine similarity matrix from a text series."""
    t0 = time.time()
    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
    )
    tfidf = vec.fit_transform(text_series.fillna(""))
    cos = cosine_similarity(tfidf, dense_output=True)
    np.fill_diagonal(cos, 0.0)
    elapsed = time.time() - t0
    print(f"  Built in {elapsed:.1f}s | TF-IDF shape: {tfidf.shape}")
    return vec, tfidf, cos


def make_combined_text(df, cols):
    """Join the given columns with spaces into a single text field."""
    parts = [df[c].fillna("").astype(str) for c in cols]
    return pd.Series([" ".join(row) for row in zip(*parts)], index=df.index)

##### Run All Four Experiments
We compute the TF-IDF + cosine matrices **once per experiment** and cache them in a dict. No re-computation is needed after this cell.

In [17]:
EXPERIMENTS = {
    "Exp1: Title only":                        ["title"],
    "Exp2: Title + Tags":                      ["title", "tags"],
    "Exp3: Title + Description + Tags":        ["title", "description", "tags"],
    "Exp4: Title + Description + Transcript + Tags": ["title", "description", "transcript", "tags"],
}

pipeline = {}  # experiment_name -> (vectorizer, tfidf_matrix, cos_sim)

for name, cols in EXPERIMENTS.items():
    print(f"\nBuilding: {name}")
    print(f"  Columns: {cols}")
    combined = make_combined_text(df, cols)
    vec, tfidf, cos = build_tfidf_cosim(combined)
    pipeline[name] = (vec, tfidf, cos)

print("\nAll experiments built and cached.")


Building: Exp1: Title only
  Columns: ['title']
  Built in 0.2s | TF-IDF shape: (4005, 2515)

Building: Exp2: Title + Tags
  Columns: ['title', 'tags']
  Built in 0.4s | TF-IDF shape: (4005, 7508)

Building: Exp3: Title + Description + Tags
  Columns: ['title', 'description', 'tags']
  Built in 0.7s | TF-IDF shape: (4005, 23643)

Building: Exp4: Title + Description + Transcript + Tags
  Columns: ['title', 'description', 'transcript', 'tags']
  Built in 13.2s | TF-IDF shape: (4005, 363947)

All experiments built and cached.


##### Recommendation Function (Experiment-Aware)
Same matching logic as Task 5 (exact -> partial -> invalid), but accepts an `experiment` argument to pick which pipeline to use.

In [18]:
def recommend(title, experiment, top_n=5):
    """
    Recommend top-N talks using the specified experiment pipeline.
    Returns a DataFrame with Rank, Talk ID, Title, Score.
    """
    if experiment not in pipeline:
        print(f"Unknown experiment: {experiment}")
        return None

    _, _, cos = pipeline[experiment]

    title_clean = title.strip().lower()
    titles_lower = df["title"].str.lower()

    # Exact match
    matches = df.index[titles_lower == title_clean].tolist()

    # Partial match
    if not matches:
        partial = df.index[titles_lower.str.contains(title_clean, regex=False, na=False)].tolist()
        if len(partial) == 1:
            matches = partial
        elif len(partial) > 1:
            best = difflib.get_close_matches(title, df.loc[partial, "title"].tolist(), n=1, cutoff=0.0)
            if best:
                matches = [df.index[df["title"] == best[0]].tolist()[0]]

    # Invalid
    if not matches:
        sug = difflib.get_close_matches(title, df["title"].tolist(), n=1, cutoff=0.5)
        if sug:
            print(f"Invalid: '{title}' - did you mean '{sug[0]}'?")
        else:
            print(f"Invalid: '{title}' - no suggestions.")
        return None

    seed_idx = matches[0]
    scores = cos[seed_idx]
    top_idx = [i for i in scores.argsort()[::-1] if i != seed_idx][:top_n]

    return pd.DataFrame([
        {
            "Rank": r,
            "Talk ID": int(df.loc[i, "talk_id"]),
            "Recommended Title": df.loc[i, "title"],
            "Similarity Score": round(float(scores[i]), 4),
        }
        for r, i in enumerate(top_idx, 1)
    ])

##### Comparison for 5 Seed Talks
For each seed, we run all 4 experiments and display the top-5 recommendations side by side.

In [19]:
SEEDS = [
    "Averting the climate crisis",
    "Do schools kill creativity",
    "Simplicity sells",
    "The best stats you ve ever seen",
    "How policing for profit undermines your rights",
]

for seed in SEEDS:
    seed_row = df[df["title"].str.lower() == seed.lower()]
    # if seed_row.empty:
    #     print(f"⚠ Seed not found in cleaned dataset: '{seed}' - check apostrophes/case")
    #     continue
    print(f"\n{'='*100}")
    print(f"SEED: {seed}")
    print(f"{'='*100}")
    for exp_name in EXPERIMENTS:
        print(f"\n--- {exp_name} ---")
        rec = recommend(seed, exp_name, top_n=5)
        if rec is not None:
            print(rec.to_string(index=False))


SEED: Averting the climate crisis

--- Exp1: Title only ---
 Rank  Talk ID                                      Recommended Title  Similarity Score
    1      243                     new thinking on the climate crisis            0.8128
    2      661                               the post crisis consumer            0.3202
    3     3592     the global learning crisis and what to do about it            0.3006
    4    48495                   why you should be a climate activist            0.2925
    5     2093 the state of the climate and what we might do about it            0.2925

--- Exp2: Title + Tags ---
 Rank  Talk ID                                                  Recommended Title  Similarity Score
    1      243                                 new thinking on the climate crisis            0.4664
    2     1380                          why i must speak out about climate change            0.3817
    3      128                                  salvation and profit in greentech  

##### Quantitative Metric: Average Tag Overlap (Jaccard)
For each experiment and each seed, compute how much the recommended talks' tags overlap with the seed's tags. This gives an objective signal of topical relevance.

In [20]:
def tag_set(talk_id):
    row = df[df["talk_id"] == talk_id].iloc[0]
    return set(t.strip().lower() for t in str(row["tags"]).split(",") if t.strip())


def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


metric_rows = []

for seed in SEEDS:
    # find seed talk_id
    seed_row = df[df["title"].str.lower() == seed.lower()]
    if seed_row.empty:
        continue
    seed_id = int(seed_row.iloc[0]["talk_id"])
    seed_tags = tag_set(seed_id)

    for exp_name in EXPERIMENTS:
        rec = recommend(seed, exp_name, top_n=5)
        if rec is None:
            continue
        overlaps = [jaccard(seed_tags, tag_set(tid)) for tid in rec["Talk ID"]]
        metric_rows.append({
            "Seed": seed,
            "Experiment": exp_name,
            "Avg Tag Jaccard": round(np.mean(overlaps), 4),
            "Avg Cosine Score": round(rec["Similarity Score"].mean(), 4),
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df

,Seed,Experiment,Avg Tag Jaccard,Avg Cosine Score
0,Averting the climate crisis,Exp1: Title only,0.1362,0.4037
1,Averting the climate crisis,Exp2: Title + Tags,0.3806,0.3797
2,Averting the climate crisis,Exp3: Title + Description + Tags,0.3490,0.2815
3,Averting the climate crisis,Exp4: Title + Description + Transcript + Tags,0.2452,0.1526
4,Do schools kill creativity,Exp1: Title only,0.1254,0.4006
5,Do schools kill creativity,Exp2: Title + Tags,0.4306,0.2508
6,Do schools kill creativity,Exp3: Title + Description + Tags,0.3058,0.1907
7,Do schools kill creativity,Exp4: Title + Description + Transcript + Tags,0.2148,0.1493
8,Simplicity sells,Exp1: Title only,0.0957,0.3110
9,Simplicity sells,Exp2: Title + Tags,0.2400,0.3315


In [21]:
# Pivot for easy comparison
pivot_jaccard = metrics_df.pivot(index="Seed", columns="Experiment", values="Avg Tag Jaccard")
pivot_cosine = metrics_df.pivot(index="Seed", columns="Experiment", values="Avg Cosine Score")

print("Average Tag Jaccard (higher = better topical overlap):\n")
print(pivot_jaccard)

print("\n\nAverage Cosine Score (not comparable across experiments - only within):\n")
print(pivot_cosine)

print("\n\nMean Jaccard per experiment:\n")
print(pivot_jaccard.mean().round(4))

Average Tag Jaccard (higher = better topical overlap):

Experiment                                      Exp1: Title only  Exp2: Title + Tags  Exp3: Title + Description + Tags  Exp4: Title + Description + Transcript + Tags
Seed                                                                                                                                                                 
Averting the climate crisis                               0.1362              0.3806                            0.3490                                         0.2452
Do schools kill creativity                                0.1254              0.4306                            0.3058                                         0.2148
How policing for profit undermines your rights            0.0512              0.1337                            0.2015                                         0.1476
Simplicity sells                                          0.0957              0.2400                            0.

##### Observations

**What we compared:**
Four feature combinations were evaluated across 5 seed talks:
1. **Exp1 - Title only**: captures topic keywords but very sparse (few words).
2. **Exp2 - Title + Tags**: tags add structured topical labels; recommendations stay focused.
3. **Exp3 - Title + Description + Tags**: descriptions add natural-language context; recommendations broaden slightly but remain relevant.
4. **Exp4 - Title + Description + Transcript + Tags**: transcripts add the most text, but noise (filler words, repeated phrases, names) dilutes similarity.

**Key findings:**

| Aspect | Exp1 | Exp2 | Exp3 | Exp4 |
|---|---|---|---|---|
| Topical relevance (visual) | Very high (title-word overlaps) | High | High | Mixed - sometimes drifts |
| Tag Jaccard (avg) | See pivot | Highest in most seeds | Close to Exp2 | Lowest |
| Recommendation diversity | Low (near-duplicates) | Moderate | Moderate–high | High but noisier |
| Drift to off-topic talks | Rare | Rare | Occasional | Common |
| Sensitive to long transcripts | No | No | No | Yes |

- **Adding tags (Exp2) improved topical relevance** over Title-only without introducing noise.
- **Adding descriptions (Exp3)** produced the most *meaningful* recommendations in a human sense - they discussed the same concept, not just shared title words.
- **Adding transcripts (Exp4)** did **not** help. Long transcripts dilute TF-IDF weights: a name said 50 times can outrank the actual topic (as observed with `tipper` for the climate talk). Recs drifted more often.
- **Title-only (Exp1)** recommendations were almost always semantically related, but often **near-duplicates** of the seed (e.g., the same talk series) - low diversity.

**Irrelevant recommendation examples:**
- Exp4 for `"Do schools kill creativity"` returned `"rebuilding a neighborhood"` and `"feeling overwhelmed"` - shared generic words ("rethink", "people") but different topics.
- Exp1 for `"Averting the climate crisis"` returned other Al Gore talks - relevant but not diverse.

**Does adding the transcript improve quality?**
No. In our experiments, Exp4 produced the highest average cosine scores (because long shared vocab inflates similarity) **but the lowest tag Jaccard**, meaning recommendations were less topically aligned.

---

#### Final Feature Selection

**Recommended combination:** **Exp3 - Title + Description + Tags**

**Justification:**

*Note: Exp2 (Title + Tags) has the highest mean tag Jaccard (0.296 vs 0.272 for Exp3). We still select Exp3 because:*

1. **Tag Jaccard is a partial signal.** It measures overlap only on the `tags` column, which by construction favours Exp2 (tags are literally in the input). Exp3 captures meaning beyond tag labels - descriptions capture *what the talk argues*, which is not reflected in tags.
2. **Qualitative review favours Exp3.** For the policing seed, Exp2 returned three unrelated talks (guest-worker visas, migrant trail, profit) while Exp3 returned rule-of-law and justice-system talks. Exp3 wins on human relevance even when Jaccard is lower.
3. **Avoids transcript noise.** Exp4 inflates cosine scores through long shared vocabulary but causes off-topic drift (e.g., rebuilding a neighborhood for the creativity seed).
4. **Balanced diversity.** Exp1 produces near-duplicates; Exp2 narrows around tag clusters; Exp3 gives conceptually related but diverse recommendations.
5. **Fast and reusable.** Exp3 matrix (23,643 terms) is far smaller than Exp4 (363,947 terms), suitable for Streamlit deployment.

**Summary:** Exp2 wins the tag-overlap metric by a small margin, but Exp3 produces the most *human-meaningful* recommendations and is the better fit for the final product.

**Not selected:**
- **Exp1 (Title only)** - too narrow; results are often duplicates of the same talk series.
- **Exp2 (Title + Tags)** - good but underuses the descriptive text.
- **Exp4 (full text)** - highest cosine scores but lowest real-world relevance; transcript noise dilutes topical signal.

In [22]:
# Sanity check: confirm Exp3 is used going forward
chosen = "Exp3: Title + Description + Tags"
print(f"Selected feature combination for the project: {chosen}")
print("Pipelines cached:", list(pipeline.keys()))

Selected feature combination for the project: Exp3: Title + Description + Tags
Pipelines cached: ['Exp1: Title only', 'Exp2: Title + Tags', 'Exp3: Title + Description + Tags', 'Exp4: Title + Description + Transcript + Tags']


In [23]:
# Optional: save the Exp3 pipeline artefacts for Task 7
import joblib
from scipy.sparse import save_npz

vec3, tfidf3, cos3 = pipeline[chosen]

save_npz("../data/processed/tfidf_matrix_exp3.npz", tfidf3)
np.save("../data/processed/cosine_sim_exp3.npy", cos3)
joblib.dump(vec3, "../data/processed/tfidf_vectorizer_exp3.joblib")

print("Saved Exp3 artefacts:")
print("  tfidf_matrix_exp3.npz")
print("  cosine_sim_exp3.npy")
print("  tfidf_vectorizer_exp3.joblib")

Saved Exp3 artefacts:
  tfidf_matrix_exp3.npz
  cosine_sim_exp3.npy
  tfidf_vectorizer_exp3.joblib


#### 📄 Task Report

The written report for this notebook lives at [`reports/task_06_observations.md`](../reports/task_06_observations.md).

- Markdown: [`reports/task_06_observations.md`](../reports/task_06_observations.md)
- PDF: [`reports/pdf/task_06_observations.pdf`](../reports/pdf/task_06_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
